In [ ]:
#package installation
# pip install datasets chromadb sentence-transformers
# pip install newsapi-python

In [ ]:
# RAG Data Ingestion Script (Persistent ChromaDB)
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings

#
# 1. Initialize Sentence Transformer model
#
model = SentenceTransformer("all-MiniLM-L6-v2")

#
# 2. Initialize Chroma with persistent storage
#
chroma = chromadb.PersistentClient(path="./chroma_store")

#
# 3. (Optional) Delete existing collection if you want a fresh start
#
try:
    chroma.delete_collection("startup_finance_rag")
except Exception:
    pass

#
# 4. Create a new collection for your startup datasets
#
collection = chroma.create_collection(name="startup_finance_rag")

#
# 5. Load datasets from Hugging Face
#
datasets = [
    "aggubandar/startup-funding-llm-data",
    "TuningAI/Startups_V2"
]

#
# 6. Iterate through datasets and encode + store in Chroma
#
for dname in datasets:
    ds = load_dataset(dname, split="train", streaming=True)
    print(f" Ingesting {dname} (streaming mode)")

    for i, row in enumerate(ds):
        # handle flexible column names across datasets
        question = (
            row.get("prompt") or
            row.get("question") or
            row.get("input") or
            ""
        )
        answer = (
            row.get("response") or
            row.get("answer") or
            row.get("output") or
            ""
        )

        text = f"Q: {question}\nA: {answer}"
        emb = model.encode(text)

        collection.add(
            ids=[f"{dname}_{i}"],
            documents=[text],
            embeddings=[emb],
            metadatas={"source": dname}
        )

#
# 7. Persist all embeddings to disk (PersistentClient saves automatically)
#
print("Ingestion complete and saved to ./chroma_store/")

#
# 8. Verify saved records
#
loaded = chromadb.PersistentClient(path="./chroma_store")
collection = loaded.get_collection("startup_finance_rag")
print("Total documents stored:", collection.count())


 Ingesting aggubandar/startup-funding-llm-data (streaming mode)
 Ingesting TuningAI/Startups_V2 (streaming mode)
Ingestion complete and saved to ./chroma_store/
Total documents stored: 35848


In [ ]:
from newsapi import NewsApiClient
import datetime
import os
from dotenv import load_dotenv

load_dotenv()

# Initialize the NewsAPI client once (reuses the same key)
news_api_key = os.getenv("NEWSAPI_KEY")
if not news_api_key:
    raise EnvironmentError("Set NEWSAPI_KEY in your .env file before running.")
news_client = NewsApiClient(api_key=news_api_key)

def fetch_latest_news(query: str, client: NewsApiClient = news_client):
    """Fetch the latest news articles related to a company or topic."""
    data = client.get_everything( ##here this calls everything with a 7 day window
        q=query,
        language="en",
        sort_by="publishedAt",
        page_size=10, ##here this is the number of articles we want to fetch
    )

    articles = []
    for a in data.get("articles", []):
        articles.append({
            "title": a.get("title"),
            "source": a.get("source", {}).get("name"),
            "url": a.get("url"),
            "description": a.get("description"),
        })

    return articles


In [23]:
##code to check the news api

if __name__ == "__main__":
    query = "dental artificial intelligence OR AI dentistry"
  # you can change this to anything like "dental startups", "Tesla", etc.
    articles = fetch_latest_news(query)

    for i, article in enumerate(articles, start=1):
        print(f"{i}. {article['title']} ({article['source']})")
        print(f"   {article['description']}")
        print(f"   URL: {article['url']}\n")

1. 'Dental revolution': Forget implants and fillings, soon you may get lab-grown tooth. What research breakthrough shows (The Times of India)
   Adults could one day grow their own replacement teeth instead of having fillings – as scientists have made a key discovery. Unlike implants and fillings, which are fixed and cannot adapt over time, a lab-grown tooth made from a patient’s own cells could integ…
   URL: https://economictimes.indiatimes.com/news/new-updates/dental-revolution-forget-implants-and-fillings-soon-you-may-get-lab-grown-tooth-what-research-breakthrough-shows/articleshow/125433481.cms

2. A pilot study of digital technology combined with case-based learning in clinical implantology training for specialists (Biomedcentral.com)
   Objective To explore the feasibility and potential benefits of combining digital technology with case-based learning (CBL) in teaching clinical knowledge and skills in implantology for oral and maxillofacial surgery specialists. Methods Six atten

In [ ]:
# LangChain ReAct agent wired with NewsAPI + Chroma vector store (Groq only)
import os
import chromadb
from langchain_core.tools import Tool
from langchain_groq import ChatGroq
from sentence_transformers import SentenceTransformer
from langchain.agents import create_react_agent
from langchain import hub
from langchain.agents import AgentExecutor

# Here using this model to embed the text
model = SentenceTransformer("all-MiniLM-L6-v2")

# This loads your embedding model and connects to your vector DB.
vector_client = chromadb.PersistentClient(path="./chroma_store")
vector_collection = vector_client.get_collection("startup_finance_rag") ##here this is the collection name where embeddings are stored


##here it loads the top k results from the vector database
def semantic_retriever(query: str, k: int = 5) -> str:
    """Return the top-k semantic matches from the startup corpus."""
    ##here model is taking the query and embedding it
    embedding = model.encode(query).tolist()
    ##here the similarity search is done
    results = vector_collection.query(query_embeddings=[embedding], n_results=k)
    ##here the results are fetched
    docs = results.get("documents", [[]])[0]
    metas = results.get("metadatas", [[]])[0]
    payloads = [
        f"Source: {meta.get('source', 'unknown')} | Evidence: {doc}"
        for doc, meta in zip(docs, metas)
    ]
    return "\n".join(payloads) if payloads else "No supporting passages found."

##Below is the function that fetches the latest news articles from the NewsAPI.
def news_lookup(query: str) -> str:
    """Fetch the latest headlines for a company/topic."""
    stories = fetch_latest_news(query)
    if not stories:
        return "No recent news found."
    return "\n\n".join(
        f"{s['title']} ({s['source']})\n{s['description']}\n{s['url']}"
        for s in stories
    )

##Tools!!
semantic_tool = Tool(
    name="SemanticEvidenceSearch", ##name of the tool
    func=semantic_retriever,
    description="Retrieve semantic matches from the Chroma vector DB for competitor/funding intel."
)

news_tool = Tool(
    name="NewsInsightLookup", ##name of the tool
    func=news_lookup,
    description="Fetch the latest news articles (max 5) for a startup, market, or technology query."
)

# Configure the Groq LLM that will drive the ReAct reasoning loop
##Here using Groq to load the llamma model to use further for React agent.
GROQ_MODEL = "llama-3.1-8b-instant"
GROQ_KEY = os.getenv("GROQ_API_KEY")
if not GROQ_KEY:
    raise EnvironmentError("Set GROQ_API_KEY in your .env file before running the agent.")

llm = ChatGroq(
    groq_api_key=GROQ_KEY,
    model_name=GROQ_MODEL,
    temperature=0, ##here this is to set the temperature to 0, so the output will be deterministic(no randomness)
    stream=False, ##here this is to disable the streaming of the output, output will be in one go
)

# -----------------------
# ReAct Agent (NEW API)
# -----------------------
##pre-built prompt design from LangChain Hub
prompt = hub.pull("hwchase17/react")


#3here this is the react agent creation
agent = create_react_agent(
    llm=llm,
    tools=[news_tool,semantic_tool],
    prompt=prompt,
)
##here this is the agent executor creation where we will be pasisng the input and it will be executed by the agent
agent_executor = AgentExecutor(
    agent=agent,
    tools=[news_tool,semantic_tool],
    verbose=True
)

# Example usage (uncomment to run)
query = """
I am planning to start an AI-agent startup and I want to understand whether my current financial situation supports this decision.
Here are my details:
• Current savings: ₹9,80,000
• Expected monthly burn (MVP + infra + salaries): ₹1,20,000
• Expected revenue in first 6 months: ₹0
• Estimated marketing spend: ₹40,000/month
• Time I can work without salary: 10 months
• Competition level: high

Given these details, tell me whether I am financially positioned to start this AI-agent startup, how long my runway will last, 
and whether the market scope justifies the financial risk.
"""
response = agent_executor.invoke({"input": query})
print(response)


/Users/krishnagupta/miniconda3/lib/python3.13/site-packages/langchain_groq/chat_models.py:143: UserWarning: WARNING! stream is not default parameter.
                    stream was transferred to model_kwargs.
                    Please confirm that stream is what you intended.
  warnings.warn(




> Entering new AgentExecutor chain...
Thought: To determine whether I am financially positioned to start this AI-agent startup, I need to assess my current financial situation and compare it with the expected expenses and revenue.

Action: NewsInsightLookup
Action Input: "AI-agent startup funding requirements"Writer upgrades agentic AI capabilities with new AI agent for enterprise work (SiliconANGLE News)
Generative artificial intelligence startup Writer Inc. launched a major expansion to its platform today with the debut of Writer Agent, a system that adds advanced capabilities and customizable autonomous automation for enterprise work. The new Writer Agent r…
https://siliconangle.com/2025/11/18/writer-upgrades-agentic-ai-capabilities-new-ai-agent-enterprise-work/

The AI startup lawmakers consulted on TikTok's and DeepSeek's privacy risks raised $14 million. Read its pitch deck. (Business Insider)
Read the Series A pitch deck for Feroot, a Canadian cybersecurity startup that builds

In [ ]:
# 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

In [ ]:
1) cross check the data you are using here because when you are using the financial data so is the data good to use or not.
2) add the mcp model here 
3) verify the model using the few shot learing 









In [ ]:


import http

multi-server-mcp-client
streamable-http
stdio
fast-mcp
langchain-mcp-adapters